# DES Y3 demonstration: how three-point shear changes IA constraints

A galaxy's observed ellipticity contains gravitational shear and an intrinsic
shape. If neighboring galaxies align with the same tidal field, their
intrinsic shapes correlate with each other and with the matter that lenses
other galaxies. An intrinsic-alignment (IA) parameter can therefore imitate
part of a cosmological change in a two-point shear correlation. A three-point
measurement weights the matter distribution differently and can separate
some of these parameter responses. This notebook examines that mechanism
with actual predictions and then a joint Fisher forecast.

The calculation keeps **400 two-point and 80 aperture-skewness rows** from
the public Y3 simulation files. It compares the nonlinear-alignment model
(NLA) with tidal alignment and tidal torquing (TATT), while varying $w_0$
and $w_a$. These are **demonstration forecasts**, not published DES constraints.
The sources undergo recorded clipping and normalization; the covariance is
fixed; exact agreement between its measurement weights and our finite-triangle
model is unvalidated. Numerical convergence is examined in
[EXAMPLE_FISHER_ACCURACY.ipynb](EXAMPLE_FISHER_ACCURACY.ipynb).

The sequence follows the Roman Fourier and LSST Y1 teaching notebooks:
settings and inputs, physical responses, Fisher construction, and numerical
checks. Expensive cells read matching saved results when available and run
the declared calculation when those results are absent.

## 1. Set up Python and the figure style

`nw` contains this project's notebook wrappers. Each expensive operation calls
the command-line driver with a named YAML file. Reused predictions are checked
against that configuration, measurement row order, covariance, physical inputs
and implementation files. Existing
files with different settings are refused rather than overwritten.

The C loops use at most four OpenMP workers. BLAS, used for matrix operations,
uses one worker. The runtime check also covers a reused notebook kernel:
environment variables alone cannot change a BLAS library already loaded.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import os
from pathlib import Path
import sys
import importlib

for variable in (
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "BLIS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "OMP_NUM_THREADS",
):
    os.environ[variable] = "1"
os.environ["OMP_THREAD_LIMIT"] = "4"
PROJECT = Path.cwd()
if not (PROJECT / "PAPER_TARGETS.md").is_file():
    raise ValueError("Use des_3pcf as this notebook's working directory.")
CORE = PROJECT.parents[1] / "external_modules" / "code" / "cosmolike_core"
sys.path.insert(0, str(PROJECT / "interface"))
sys.path.insert(0, str(CORE))
import numpy as np
import matplotlib
from matplotlib import pyplot as plt
from threadpoolctl import threadpool_info, threadpool_limits
import cosmolike_des_3pcf_fisher_wrappers as nw

blas_limit = threadpool_limits(limits=1, user_api="blas")
for library in threadpool_info():
    if library["user_api"] == "blas" and library["num_threads"] != 1:
        raise RuntimeError("A loaded BLAS library has more than one worker.")
plots = importlib.import_module(name="cosmolike_notebook_utils.3pcf")

In [ ]:
# Figure style of the whole notebook: the shared plotting functions never change
# rcParams, so these lines set every font, tick and grid default.
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
matplotlib.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
matplotlib.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['xtick.top'] = False
matplotlib.rcParams['ytick.right'] = False
matplotlib.rcParams['axes.edgecolor'] = 'black'
matplotlib.rcParams['axes.linewidth'] = '1.0'
matplotlib.rcParams['axes.labelsize'] = 'medium'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.linewidth'] = '0.0'
matplotlib.rcParams['grid.alpha'] = '0.18'
matplotlib.rcParams['grid.color'] = 'lightgray'
matplotlib.rcParams['legend.labelspacing'] = 0.77
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.format'] = 'pdf'
matplotlib.rcParams["figure.dpi"] = 90

## 2. Read the sources and the measurement

The redshift distribution $n_i(z)$ gives the relative number of source
galaxies in bin $i$ per unit redshift, with integral one. It controls both
the lensing and IA weights. The original simulation FITS has negative
midpoint samples. The authorized demonstration clips those samples to zero
and normalizes each bin by a trapezoidal integral over the supplied midpoint
redshifts. This declared input alteration is not a source recalibration
and does not supply an uncertainty model.

The JSON sidecar records original and derived file hashes, removed negative
areas and normalization factors. The wrapper verifies the derived FITS hash
before plotting. Physical-table preparation uses a shape-preserving cubic (PCHIP) through
the processed midpoint samples, with zeros at the measured outer edges and
a zero plateau to the background limit. The last source cell ends at
$z=3.0$; extending its nonzero last midpoint toward the background limit
$z=3.1$ would create a populated tail absent from that support. The
reconstruction is sampled on 9601 uniform nodes and normalized again. It
preserves the chosen support, not each histogram cell probability; it is a
declared source model, not a calibrated source correction. The plotted
curves below are the processed native samples. The [source study](tests/studies/source_policy_validation.md)
compares their reconstructions and separate source, kernel and radial grids.

In [ ]:
workflows = {
    "nla": nw.demonstration_workflow(model="nla"),
    "tatt": nw.demonstration_workflow(model="tatt"),
}
sources = nw.source_report(workflow=workflows["tatt"])
print("Midpoint integrals:", sources["midpoint_integrals"])
print("Policy:", sources["preprocessing"]["policy"])
print("Per-bin record:", sources["preprocessing"]["bins"])
source_figure, source_axes = nw.plot_sources(report=sources)
plt.show()

The two-point rows are $\xi_+(\theta)$ and $\xi_-(\theta)$, correlations of
two projected shapes separated by $\theta$. The three-point rows are
$\langle M_{\rm ap}^3(R)\rangle$, the skewness of shear integrated with a
compensated aperture filter. The aperture radius $R$ differs from the
triangle-side cut used to estimate the moment: here $R=7,14,25,40$ arcmin,
while the model integrates triangle sides from $8$ to $80$ arcmin. Exact
measurement bins and weights still need to be matched.

The correlation matrix is $C_{ab}/\sqrt{C_{aa}C_{bb}}$. Its off-diagonal
two-point/three-point rectangle shows why their Fisher matrices cannot simply
be added. This notebook retains the cross-covariance and all rows, with no
hidden scale selection.

In [ ]:
public_inputs = nw.verify_public_inputs()
print("Public source:", public_inputs["repository"])
print("Revision:", public_inputs["commit"])
print("File checks:", public_inputs["files"])
rows, covariance, counts = workflows["tatt"].public_covariance()
print("Rows:", counts, "matrix shape:", covariance.shape)
plots.plot_correlation(
    covariance=covariance,
    block_sizes=[counts["xip"], counts["xim"], counts["map3"]],
    block_labels=[r"$\xi_+$", r"$\xi_-$", r"$M_{\rm ap}^3$"],
)

## 3. Compare actual IA sign responses

NLA uses tidal-alignment amplitude $A_1$ and redshift exponent $\eta_1$.
TATT adds a quadratic tidal response with amplitude $A_2$, exponent $\eta_2$,
and density-weighting coefficient $b_{\rm TA}$. Opposite signs of
$(A_1,A_2)$ can give similar two-point curves while changing aperture
skewness. The choices $(-0.2,0.4)$ and $(0.2,-0.4)$ follow the question
examined in Figure 9 of
[Gomes et al., arXiv:2601.09133](https://arxiv.org/abs/2601.09133).
Other parameters remain those of the TATT demonstration, including its
Table 1 IA fiducial values and specified nonlinear-power prescription.

The next cell writes two explicit JSON configurations and reads or computes
the fiducial plus both sign choices. These are actual $\xi_\pm$ and
finite-triangle $M_{\rm ap}^3$ predictions. Four auto-bin groups are displayed
for legibility; all cross-bin rows enter the forecast. This tests the
mechanism, **not an absolute reproduction of Figure 9**. Our input combination,
triangle weights and starting $L_{\max}=M_{\max}=20$ require comparison with
the paper's finer $L_{\max}=50$, $M_{\max}=100$ calculation.

In [ ]:
sign_workflows, sign_labels = nw.ia_sign_workflows(
    workflow=workflows["tatt"],
)
sign_results = []
for case in sign_workflows:
    result, declaration = case.ensure_result(command="evaluate")
    sign_results.append(result)
for kind in ("xip", "map3"):
    response_figure, response_axes = nw.plot_mean_responses(
        results=sign_results, labels=sign_labels, kind=kind,
    )
    plt.show()

Residual panels show $100[X-X_{\rm fid}]/\max|X_{\rm fid}|$, with one scale
for the full angular curve in each bin. This is not a pointwise percentage
and remains defined at a zero crossing. Similar curves in selected panels
do not establish a degeneracy of the full data vector, which includes every
bin, angle and correlation.

In [ ]:
for kind in ("xip", "map3"):
    response_figure, response_axes = nw.plot_mean_responses(
        results=sign_results, labels=sign_labels, kind=kind, residual=True,
    )
    plt.show()

## 4. What the joint Fisher matrix measures

For mean $\mu(\boldsymbol p)$ and fixed covariance $C$, let
$D_{a\alpha}=\partial\mu_a/\partial p_\alpha$. A local Gaussian forecast is
$$F_{\alpha\beta}=D_{a\alpha}(C^{-1})_{ab}D_{b\beta}
+F^{\rm prior}_{\alpha\beta},$$
with repeated data indices summed. Diagonal entries of $F^{-1}$ are squared
**marginalized** errors: every other varied parameter can move. In contrast,
$1/\sqrt{F_{\alpha\alpha}}$ holds those parameters fixed. An unresolved
matrix produces a rank diagnostic, never a pseudoinverse-based error bar.

Partition the data into two- and three-point rows, $(d_2,d_3)$. Removing the
part of the three-point fluctuation predicted by $d_2$ gives
$$S=C_{33}-C_{32}C_{22}^{-1}C_{23},\qquad
E=D_3-C_{32}C_{22}^{-1}D_2.$$
The likelihood information is then
$$F_{\rm joint}=F_2+F_{3|2},\qquad F_{3|2}=E^{\mathsf T}S^{-1}E.$$
The added term is **conditional on the two-point measurement**. It is not
an independently computed three-point Fisher matrix. Priors are added once
to each experiment; the saved arrays retain $S$, $E$ and all Fisher pieces.

Both models use the same calibration and photo-z prior widths. NLA and TATT
vary different IA spaces, so their cosmological contours describe different
marginalizations. Finite-difference bounds are not Gaussian priors.

## 5. Read or run the demonstration forecasts

Each YAML lists absolute parameter displacements $h$. The driver computes
five-point derivatives at $h$ and $h/2$, retains both, and keeps one covariance
fixed across every displacement. It varies $w_0$ and $w_a$ without Gaussian
priors. The printed mean count is a work count, not a runtime estimate.

These cells run the forecast if its saved result is absent. The configuration
hash includes comments; an edited YAML needs a matching result. A hash match
does not establish numerical convergence. Inspect the rank and step checks
before interpreting the error bars, then use the accuracy notebook to test
quadrature and interpolation separately. For each parameter, the printed
step percentage is $100\|D_h-D_{h/2}\|_C/\|D_{h/2}\|_C$, with
$\|v\|_C^2=v^{\mathsf T}C^{-1}v$. Using the complete covariance keeps
the much smaller raw aperture amplitudes from disappearing behind the
two-point amplitudes. Step agreement alone does not establish grid convergence.

In [ ]:
forecasts = {}
declarations = {}
for model in ("nla", "tatt"):
    inspection = workflows[model].inspect(command="fisher")
    print(model.upper(), inspection["description"])
    print("Parameters:", inspection["varied_parameters"])
    print("Mean evaluations:", inspection["distinct_mean_evaluations"])
    print("Gaussian priors:", inspection["gaussian_prior_sigmas"])
    forecasts[model], declarations[model] = workflows[model].ensure_result(
        command="fisher",
    )
    summary = nw.fisher_summary(
        result=forecasts[model], metadata=declarations[model],
    )
    print("Rank:", summary["rank_status"])
    print("Conditional identity relative error:",
          summary["information_identity_relative_error"])
    for name, change in zip(
        summary["names"], summary["derivative_covariance_relative_change"],
    ):
        print(f"{name}: covariance-weighted h/h2 change = {100.0 * change:.6g}%")

### IA errors within each model

Each point shows $\sigma_{\rm joint}(p)/\sigma_{\xi}(p)$ for one IA
parameter. A value below one means a smaller marginalized error, with the
same prior counted once. The two figures have different IA spaces. They do
not measure the cost of adding a parameter with all other assumptions fixed.
If either matrix has unresolved directions, the wrapper refuses the plot.

In [ ]:
for model in ("nla", "tatt"):
    print("IA model:", model.upper())
    error_figure, error_axis = nw.plot_ia_errors(result=forecasts[model])
    plt.show()

### Common cosmological parameters after marginalizing IA

The contours compare joint NLA and TATT in $(\Omega_m,\sigma_8,w_0,w_a)$.
Each full covariance is calculated before selecting these coordinates.
Selecting a Fisher submatrix first would fix omitted nuisance parameters
and overstate information when IA, photo-z and calibration are uncertain.
The contours are local Gaussians, without clipping at parameter bounds.

In [ ]:
contours = nw.plot_common_contours(
    results=[forecasts["nla"], forecasts["tatt"]],
    metadata=[declarations["nla"], declarations["tatt"]],
    labels=["NLA joint demonstration", "TATT joint demonstration"],
    parameters=[
        "omega_m",
        "sigma8",
        "w0",
        "wa",
    ],
)
plt.show()

### A derived amplitude and within-model information gain

The commonly used amplitude is $S_8=\sigma_8\sqrt{\Omega_m/0.3}$.
Its local covariance follows from the full marginalized
$(\Omega_m,\sigma_8)$ covariance by $C'=JCJ^{\mathsf T}$, with
$\partial S_8/\partial\Omega_m=S_8/(2\Omega_m)$ and
$\partial S_8/\partial\sigma_8=\sqrt{\Omega_m/0.3}$. Dropping the
$\Omega_m$--$\sigma_8$ cross term would give the wrong error.

For each model, the cell reports $\sigma(S_8)$ and the joint-to-two-point
ratio of $1/\sqrt{\det C_{\Omega_m,S_8}}$. The common fiducial Jacobian
cancels from this ratio, so it equals the ratio in
$(\Omega_m,\sigma_8)$. These are numerical diagnostics of this demonstration.
The paper's quoted percentage improvements are not acceptance thresholds:
its priors, cuts and likelihood differ. Numerical accuracy must be assessed
before any FoM is used as a scientific result.

In [ ]:
for model in ("nla", "tatt"):
    amplitude_summary = nw.derived_amplitude_summary(
        result=forecasts[model], metadata=declarations[model],
    )
    print(model.upper(), "fiducial S8:", amplitude_summary["fiducial_s8"])
    for probe in ("two_point", "joint"):
        print(probe, "sigma(S8):", amplitude_summary[probe]["sigma_s8"])
    print("Joint / two-point (Omega_m,S8) FoM:",
          amplitude_summary["joint_over_two_point_fom"])

## 6. Interpret the result with its assumptions

A smaller IA error or changed degeneracy direction shows the local prediction
of this mean and covariance. It does not establish baryon insensitivity,
exact agreement with the public aperture estimator, or the adequacy of
omitting nonlinear-IA covariance changes. All 400 two-point rows include
scales removed by the published analysis. Both mean models reuse the public
covariance; connected TATT covariance responses are absent.

Before quoting an improvement, use
[EXAMPLE_FISHER_ACCURACY.ipynb](EXAMPLE_FISHER_ACCURACY.ipynb) to inspect
numerical noise and refine the mean independently of derivative steps.
Match triangle support and averaging to the measurement, apply explicit
scientific cuts, and examine remaining physical assumptions. Finite-simulation
precision corrections also require the independent realization structure,
not merely a patch count. That issue is developed in
[EXAMPLE_COVARIANCE_PME.ipynb](EXAMPLE_COVARIANCE_PME.ipynb).